# val_02_lickometer — validating lickometer licks against Lightning-Pose

A lick is detected in the pose data when the tongue comes within a threshold distance of either
spout. Detected licks are matched against the lickometer across three parameters: the spatial
threshold, a refractory filter, and the coincidence window used to call two events the same lick.

Counts and fractions are named for the list they describe — `n_pose_only`, `n_lickometer_only`,
`pose_only_frac`, `lickometer_only_frac`. The convention is set out in §3.

Session: `behavior_716325_2024-05-31_10-31-14`.

1. Setup
2. Load session data
3. Lick detection, worked example
4. Parameter sweep
5. Sweep figures
6. Inter-lick intervals
7. Metric curves at the chosen parameters
8. Individual events
9. Labeled video clips

Code Ocean only — needs the per-session `intermediate_data/` parquets and the labeled video.

## 1. Setup

In [ ]:
%matplotlib inline
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from aind_dynamic_foraging_behavior_video_analysis.kinematics.tongue_kinematics_utils import (
    mask_keypoint_data,
)
from aind_dynamic_foraging_behavior_video_analysis.kinematics.tongue_lickometer_utils import (
    detect_licks,
    filter_timestamps_refractory,
    calculate_metrics,
    calculate_metrics_witheventkeys,
    extract_clips_ffmpeg_encode,
)
from aind_dynamic_foraging_behavior_video_analysis.ephys.tongue_ephys import find_session_dir

In [ ]:
if Path("/root/capsule").exists():
    ENV     = "codeocean"
    SCRATCH = Path("/root/capsule/scratch")
    DATA    = Path("/root/capsule/data")
else:
    ENV     = "local"
    SCRATCH = Path("/Users/mib/Documents/Code/kinematics_analysis/data/for_local").parent
    DATA    = SCRATCH

IS_CO    = ENV == "codeocean"
FIG_DIR  = SCRATCH / "figures" / "val_02_lickometer"
SAVE_FIG = False

SESSION_DIR     = SCRATCH / "session_analysis_mlk"
EXAMPLE_SESSION = "behavior_716325_2024-05-31_10-31-14"
CONF            = 0.8      # masking confidence threshold

print("ENV={}".format(ENV))

## 2. Load session data

From the per-session `intermediate_data/` parquets written by
`tongue_analysis.run_batch_analysis`. `time_in_session` comes from `tongue_kins`, which
`kinematics_filter` returns with the same rows in the same order as the raw keypoints, and is
the same time base as `nwb_df_licks['timestamps']`.

`tongue_tip` is called `tongue_tip_center` in the current keypoint set.

In [ ]:
if not IS_CO:
    print("[skip] Code Ocean only: needs session_analysis_mlk/<session>/intermediate_data/.")
else:
    inter = find_session_dir(EXAMPLE_SESSION, roots=[SESSION_DIR]) / "intermediate_data"

    keypoint_dfs = {
        key: pd.read_parquet(inter / "kps_raw_{}.parquet".format(key))
        for key in ["tongue_tip_center", "spout_l", "spout_r"]
    }
    time_in_session = pd.read_parquet(
        inter / "tongue_kins.parquet", columns=["time_in_session"])["time_in_session"].values

    # Extract tongue dataframe and mask
    tongue_masked = mask_keypoint_data(keypoint_dfs, "tongue_tip_center",
                                       confidence_threshold=CONF)
    # kps_raw_* already carries `time`, which is video-relative. Overwrite it with the
    # go-cue-relative session time, the base all_licks is on.
    tongue_masked["time"] = time_in_session

    # All licks, both spouts
    all_licks = np.sort(pd.read_parquet(inter / "nwb_df_licks.parquet")["timestamps"].to_numpy())

    # NB R and L are switched due to mislabeling in the training data
    mean_spoutL = np.mean(keypoint_dfs["spout_r"][["x", "y"]], 0)
    mean_spoutR = np.mean(keypoint_dfs["spout_l"][["x", "y"]], 0)

    print("Frames: {:,}   tracked at conf >= {}: {:,}".format(
        len(tongue_masked), CONF, int(tongue_masked["x"].notna().sum())))
    print("Lickometer licks: {:,}".format(len(all_licks)))

## 3. Lick detection, worked example

1. Detect licks in the pose data using a threshold on distance to the spout.
2. Determine sensitivity and specificity of the lickometer with respect to those licks.

### Counts and their names

`calculate_metrics(a, b, w)` returns three counts: events matched across the two lists within
`w`, events in `b` with no match, and events in `a` with no match. Pose licks are passed as `a`
and lickometer licks as `b` everywhere below, so the counts are `n_matched`, `n_lickometer_only`
and `n_pose_only`.

- `n_matched + n_pose_only` is the number of pose licks.
- `n_matched + n_lickometer_only` is the number of lickometer licks.

The four fractions are named for the list they are a fraction of: `pose_matched_frac` and
`pose_only_frac` sum to 1 over the pose licks, `lickometer_matched_frac` and
`lickometer_only_frac` sum to 1 over the lickometer licks. `f1_score` is symmetric in the two
lists.

### 30 px spatial threshold, 0.05 s refractory filter, 0.1 s coincidence window

In [ ]:
# Detect licks based on threshold crossing (pixels) close to either spout
LP_licks_temp = detect_licks(tongue_masked, mean_spoutL, mean_spoutR, 30)

# Filter licks that happen shortly after another lick (threshold fluctuation effect)
LP_licks_temp = filter_timestamps_refractory(LP_licks_temp, 0.05)

# Match the two lists with a 0.1 s coincidence window
n_matched, n_lickometer_only, n_pose_only = calculate_metrics(LP_licks_temp, all_licks, 0.1)

n_pose = n_matched + n_pose_only
n_lickometer = n_matched + n_lickometer_only

# Of the pose licks, how many have a lickometer counterpart?
pose_matched_frac = n_matched / n_pose
pose_only_frac = n_pose_only / n_pose

# Of the lickometer licks, how many have a pose counterpart?
lickometer_matched_frac = n_matched / n_lickometer
lickometer_only_frac = n_lickometer_only / n_lickometer

print("pose licks with no lickometer match: {:.2f}".format(pose_only_frac))
print("lickometer licks with no pose match: {:.2f}".format(lickometer_only_frac))

## 4. Parameter sweep

In [ ]:
# Sweep the detection and matching parameters
# 1. spatial thresholds for lick detection (in pixels)
# 2. time thresholds for overlap between pose and lickometer coincidence (ie whether 'same
#    event' or not)
# 3. time thresholds for 'refractory filter' -- removes spurious detection of licks due to
#    oscillation around the spatial lick detection threshold
spatial_thresholds = np.arange(10, 51, 5)
time_thresholds = np.arange(0.005, 0.251, 0.005)
t_refractory_values = np.arange(0, 0.11, 0.01)

results = []

for spatial_threshold in spatial_thresholds:
    # Detect licks based on spatial threshold
    LP_licks_temp = detect_licks(tongue_masked, mean_spoutL, mean_spoutR, spatial_threshold)

    for t_refractory in t_refractory_values:
        # Filter detected licks based on the current t_refractory value
        LP_licks_filtered = filter_timestamps_refractory(LP_licks_temp, t_refractory)

        for time_threshold in time_thresholds:
            n_matched, n_lickometer_only, n_pose_only = calculate_metrics(
                LP_licks_filtered, all_licks, time_threshold)

            n_pose = n_matched + n_pose_only
            n_lickometer = n_matched + n_lickometer_only

            results.append({
                'spatial_threshold': spatial_threshold,
                'time_threshold': time_threshold,
                't_refractory': t_refractory,
                'n_matched': n_matched,
                'n_lickometer_only': n_lickometer_only,
                'n_pose_only': n_pose_only,
                'pose_matched_frac': n_matched / n_pose if n_pose > 0 else 0,
                'pose_only_frac': n_pose_only / n_pose if n_pose > 0 else 0,
                'lickometer_matched_frac': n_matched / n_lickometer if n_lickometer > 0 else 0,
                'lickometer_only_frac': n_lickometer_only / n_lickometer if n_lickometer > 0 else 0,
                # Harmonic mean of the two matched fractions; symmetric in the two lists
                'f1_score': (2 * n_matched) / (2 * n_matched + n_lickometer_only + n_pose_only),
            })

results_df = pd.DataFrame(results)

In [ ]:
# What are the top results?
n = 5
test_df = results_df
# # modify below to select subset of parameters, eg
# test_df = results_df.query('time_threshold <= 0.1 and t_refractory <= 0.05')

top_n_rows = test_df.nlargest(n, 'f1_score')
top_n_rows

## 5. Sweep figures

In [ ]:
# Figure 1: heatmaps comparing each parameter (time threshold, spatial threshold,
# refractory filter time)

df = results_df.copy()
df['time_threshold'] = df['time_threshold'].round(3)  # due to plotting issues with floats

# Color range for the heatmaps
cmap_range_min = 0.5
cmap_range_max = 1

fig, axs = plt.subplots(1, 3, figsize=(16, 4))

# Heatmap 1: spatial_threshold vs. time_threshold
heatmap_data1 = df.pivot_table(values='f1_score', index='spatial_threshold',
                               columns='time_threshold')
sns.heatmap(heatmap_data1, ax=axs[0], cmap='YlGnBu', cbar_kws={'label': 'F1 Score'},
            vmin=cmap_range_min, vmax=cmap_range_max)

max_value1 = heatmap_data1.max().max()
spatial_threshold1, time_threshold1 = heatmap_data1.stack().idxmax()
print("Heatmap 1 Max - Spatial Threshold: {}, Time Threshold: {}, F1 Score: {:.2f}".format(
    spatial_threshold1, time_threshold1, max_value1))

axs[0].set_title('Spatial Threshold vs. Time Threshold')
axs[0].set_xlabel('Time Threshold')
axs[0].set_ylabel('Spatial Threshold')

# Heatmap 2: spatial_threshold vs. t_refractory
heatmap_data2 = df.pivot_table(values='f1_score', index='spatial_threshold',
                               columns='t_refractory')
sns.heatmap(heatmap_data2, ax=axs[1], cmap='YlGnBu', cbar_kws={'label': 'F1 Score'},
            vmin=cmap_range_min, vmax=cmap_range_max)

max_value2 = heatmap_data2.max().max()
spatial_threshold2, t_refractory2 = heatmap_data2.stack().idxmax()
print("Heatmap 2 Max - Spatial Threshold: {}, T Refractory: {}, F1 Score: {:.2f}".format(
    spatial_threshold2, t_refractory2, max_value2))

axs[1].set_title('Spatial Threshold vs. T Refractory')
axs[1].set_xlabel('T Refractory')
axs[1].set_ylabel('Spatial Threshold')

# Heatmap 3: time_threshold vs. t_refractory
heatmap_data3 = df.pivot_table(values='f1_score', index='time_threshold',
                               columns='t_refractory')
sns.heatmap(heatmap_data3, ax=axs[2], cmap='YlGnBu', cbar_kws={'label': 'F1 Score'},
            vmin=cmap_range_min, vmax=cmap_range_max)

max_value3 = heatmap_data3.max().max()
time_threshold3, t_refractory3 = heatmap_data3.stack().idxmax()
print("Heatmap 3 Max - Time Threshold: {}, T Refractory: {}, F1 Score: {:.2f}".format(
    time_threshold3, t_refractory3, max_value3))

axs[2].set_title('Time Threshold vs. T Refractory')
axs[2].set_xlabel('T Refractory')
axs[2].set_ylabel('Time Threshold')

plt.tight_layout()
plt.show()


# Figure 2: interaction plots for the two unmatched fractions and the F1 score, over time
# threshold and spatial threshold

fig, axs = plt.subplots(1, 3, figsize=(16, 4))

grouped_df = results_df.groupby(['time_threshold', 'spatial_threshold']).mean().reset_index()

unique_spatial_thresholds = grouped_df['spatial_threshold'].unique()
cmap = plt.get_cmap('viridis', len(unique_spatial_thresholds))

panels = [
    ('pose_only_frac',
     'Pose licks with no lickometer match',
     'pose_only_frac = n_pose_only / n_pose'),
    ('lickometer_only_frac',
     'Lickometer licks with no pose match',
     'lickometer_only_frac = n_lickometer_only / n_lickometer'),
    ('f1_score',
     'F1 Score',
     'f1 = 2*n_matched / (2*n_matched + n_lickometer_only + n_pose_only)'),
]

for ax, (col, title, ylabel) in zip(axs, panels):
    for i, spatial_threshold in enumerate(unique_spatial_thresholds):
        filtered_df = grouped_df[grouped_df['spatial_threshold'] == spatial_threshold]
        ax.plot(filtered_df['time_threshold'], filtered_df[col],
                label='Spatial: {}'.format(spatial_threshold), color=cmap(i))
    ax.set_title(title)
    ax.set_xlabel('Time Threshold (s)')
    ax.set_ylabel(ylabel)
    ax.grid()

axs[2].legend(title='Spatial Threshold (pixels)', bbox_to_anchor=(1.05, 1), loc='upper left')

fig.suptitle('Interaction of Spatial Threshold and Time Threshold', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# Optional: heatmap facet grid of F1 score, comparing spatial threshold and time
# threshold for each t_refractory value

g = sns.FacetGrid(results_df, col="t_refractory", col_wrap=3, height=3, aspect=1)

t_refractory_levels = results_df['t_refractory'].unique()
cbar_bool = False
for k, t_refractory in enumerate(t_refractory_levels):
    filtered_df = results_df[results_df['t_refractory'] == t_refractory].copy()
    filtered_df['time_threshold'] = filtered_df['time_threshold'].round(3)

    pivot_data = filtered_df.pivot_table(index="spatial_threshold",
                                         columns="time_threshold",
                                         values="f1_score",
                                         aggfunc='mean')

    ax = g.axes[k]

    if k == len(t_refractory_levels) - 1:
        cbar_bool = True

    sns.heatmap(pivot_data, ax=ax, cmap='YlGnBu',
                annot=False, linewidths=.5,
                vmin=0.8, vmax=1.0,
                cbar=cbar_bool,
                cbar_kws={'label': 'f1_score'})

    ax.set_title('Refractory Time: {:.2f} s'.format(t_refractory))

g.set_axis_labels('Time Threshold (s)', 'Spatial Threshold (pixels)')
g.set_titles(col_template='Refractory Time: {col_name:.2f} s')

plt.subplots_adjust(top=0.9)
plt.subplots_adjust(wspace=0.1, hspace=0.3)
g.fig.suptitle('F1 Score Heatmaps by Spatial Threshold and Time Threshold', fontsize=12)

plt.show()

### Interpretation

- Unbiased search for best parameter would indicate pixel_threshold of 35.
- Also indicates maximizing both the refractory filter value, and the overlap threshold value,
  produces best performance.
- However, unsure if this is truly warranted -- we want to avoid detecting two licks as 'the
  same' when in fact they are not.
- --> pursue data-driven parameter selection.

## 6. Inter-lick intervals

In [ ]:
# Motivate the refractory filter by analyzing the inter-lick interval distribution
from matplotlib import cm
import matplotlib.patches as patches

fig, axs = plt.subplots(1, 2, figsize=(8, 4), sharey=False)

cmap = cm.YlGnBu(np.linspace(0, 1, len(spatial_thresholds)))

for i, spatial_threshold in enumerate(spatial_thresholds):
    LP_licks_temp = detect_licks(tongue_masked, mean_spoutL, mean_spoutR, spatial_threshold)

    ILIs = np.diff(LP_licks_temp)
    # Calculate and plot the CDF
    sorted_ILIs = np.sort(ILIs)
    cdf = np.arange(1, len(sorted_ILIs) + 1) / len(sorted_ILIs)
    for ax in axs:
        ax.plot(sorted_ILIs, cdf, label='Threshold: {:.2f}'.format(spatial_threshold),
                color=cmap[i], alpha=0.7)

axs[0].set_xlim(0, 5)
axs[0].set_ylim(0, 1)
axs[1].set_xlim(0, 0.5)
axs[1].set_ylim(0, 0.8)

fig.suptitle('CDF of Inter-Lick Intervals Across Spatial Thresholds')
fig.text(0.5, 0.04, 'Inter-Lick Interval (seconds)', ha='center')
fig.text(0.04, 0.5, 'Cumulative Probability', va='center', rotation='vertical')

# Mark the region that the right panel zooms into
rect = patches.Rectangle((0, 0), 0.5, 0.8, linewidth=2, edgecolor='r', facecolor='none')
axs[0].add_patch(rect)

axs[0].legend()
axs[0].grid()
axs[1].grid()

for side in ['bottom', 'top', 'right', 'left']:
    axs[1].spines[side].set_color('red')

plt.show()

### Interpretation

- Each spatial threshold results in some ILIs that are less than 100ms, faster than a lick can
  occur.
- Let's compare to the lickometer event times.
- From before, we know we want to use threshold of 35 pixels.

In [ ]:
for spatial_threshold in [35, 30]:
    LP_licks = detect_licks(tongue_masked, mean_spoutL, mean_spoutR, spatial_threshold)

    plt.figure(figsize=(6, 4))
    plt.hist(np.diff(LP_licks), bins=100, range=[0, .5], alpha=0.8, density=True)
    plt.hist(np.diff(all_licks), bins=100, range=[0, .5], alpha=0.8, density=True)
    plt.title('Histogram of ILIs: {} pixel threshold'.format(spatial_threshold))
    plt.xlabel('Time (s)')
    plt.ylabel('Density')
    plt.legend(['Lightning Pose', 'Lickometer'])
    plt.show()

## 7. Metric curves at the chosen parameters

### Interpretation

- This seems to motivate filter of licks within 100ms of another lick.
- Data seem cleaner for threshold of 30 pixels -- I wonder if ILIs around 100ms are being
  shifted 'longer', as they need to get closer to the spout.
- --> would imply these are real events -- and by moving the threshold out further from mouth
  (closer to spout), we are gaining separation between 'real' licks and licks identified due to
  oscillation around the threshold.
- Motivates combination of 30 pixel, 100ms refractory filter.

In [ ]:
# Set specific values for spatial threshold and refractory period
spatial_threshold = 30
t_refractory = 0.1
time_thresholds = np.arange(0.005, 0.251, 0.001)

results = []

LP_licks_temp = detect_licks(tongue_masked, mean_spoutL, mean_spoutR, spatial_threshold)
LP_licks_filtered = filter_timestamps_refractory(LP_licks_temp, t_refractory)

for time_threshold in time_thresholds:
    n_matched, n_lickometer_only, n_pose_only = calculate_metrics(
        LP_licks_filtered, all_licks, time_threshold)

    n_pose = n_matched + n_pose_only
    n_lickometer = n_matched + n_lickometer_only
    f1_denom = 2 * n_matched + n_lickometer_only + n_pose_only

    results.append({
        'spatial_threshold': spatial_threshold,
        'time_threshold': time_threshold,
        't_refractory': t_refractory,
        'n_matched': n_matched,
        'n_lickometer_only': n_lickometer_only,
        'n_pose_only': n_pose_only,
        'pose_matched_frac': n_matched / n_pose if n_pose > 0 else 0,
        'pose_only_frac': n_pose_only / n_pose if n_pose > 0 else 0,
        'lickometer_matched_frac': n_matched / n_lickometer if n_lickometer > 0 else 0,
        'lickometer_only_frac': n_lickometer_only / n_lickometer if n_lickometer > 0 else 0,
        'f1_score': (2 * n_matched) / f1_denom if f1_denom > 0 else 0,
    })

results_parameterized_df = pd.DataFrame(results)

In [ ]:
fig, ax1 = plt.subplots()

ax1.plot(results_parameterized_df['time_threshold'],
         results_parameterized_df['pose_only_frac'],
         label='Pose licks with no lickometer match', color='b')
ax1.plot(results_parameterized_df['time_threshold'],
         results_parameterized_df['lickometer_only_frac'],
         label='Lickometer licks with no pose match', color='r')
ax1.plot(results_parameterized_df['time_threshold'],
         results_parameterized_df['f1_score'],
         label='F1 Score', color='g')

ax1.set_xlabel('Time Threshold (s)')
ax1.set_ylabel('Fraction')
ax1.tick_params(axis='y', labelcolor='k')
ax1.legend(loc='center right')
ax1.grid()

plt.title('Pixel Threshold: 30 | Refractory Filter: 0.1 s')
plt.show()

### Final stats

100 ms seems like a safe time overlap threshold --> not getting any further improvement in
F1 score.

In [ ]:
chosen_row = results_parameterized_df.query('time_threshold == 0.1')
chosen_row

## 8. Individual events

`calculate_metrics_witheventkeys` returns the same three counts plus one classified table per
input list. Pose licks are passed first, so `LP_licks_classified` is the pose table and
`licko_licks_classified` the lickometer table.

Its `Status` strings are fixed by the library (`True Positive` / `False Positive` /
`False Negative`). Under this call order, `False Positive` in the lickometer table marks a
lickometer lick with no pose match, and `False Negative` in the pose table marks a pose lick
with no lickometer match.

In [ ]:
# Pull out the individual unmatched events and analyze them.
# Parameters from above: 30 px spatial threshold, 0.1 s refractory filter, 0.1 s window.
LP_licks = detect_licks(tongue_masked, mean_spoutL, mean_spoutR, 30)
LP_licks = filter_timestamps_refractory(LP_licks, 0.1)
all_licks = filter_timestamps_refractory(all_licks, 0.1)

n_matched, n_lickometer_only, n_pose_only, LP_licks_classified, licko_licks_classified = (
    calculate_metrics_witheventkeys(LP_licks, all_licks, time_window=0.1))

print("matched:          {:,}".format(n_matched))
print("lickometer only:  {:,}".format(n_lickometer_only))
print("pose only:        {:,}".format(n_pose_only))

In [ ]:
# Distance to the nearer spout, per frame

spoutL_pos = np.array([mean_spoutL['x'], mean_spoutL['y']])
spoutR_pos = np.array([mean_spoutR['x'], mean_spoutR['y']])

# Only rows with both coordinates present
mask = tongue_masked[['x', 'y']].notna().all(axis=1)
valid_rows = tongue_masked[mask]

distances_L = np.linalg.norm(valid_rows[['x', 'y']].to_numpy() - spoutL_pos, axis=1)
distances_R = np.linalg.norm(valid_rows[['x', 'y']].to_numpy() - spoutR_pos, axis=1)

tongue_masked.loc[mask, 'distance_to_left_spout'] = distances_L
tongue_masked.loc[mask, 'distance_to_right_spout'] = distances_R

spout_cols = ['distance_to_left_spout', 'distance_to_right_spout']
tongue_masked['nearest_spout_distance'] = tongue_masked[spout_cols].min(axis=1)

# idxmin over the rows that have a distance. Running it over every row hits the all-NaN
# untracked frames, which pandas deprecates and which returned NaN -- and NaN then fell
# through the Left/Right test to 'Right', labelling untracked frames as right-spout.
# Those frames are left unlabelled here instead.
nearest = tongue_masked.loc[mask, spout_cols].idxmin(axis=1)
tongue_masked['nearest_spout'] = pd.Series(
    np.where(nearest == 'distance_to_left_spout', 'Left', 'Right'), index=nearest.index)

tongue_masked = tongue_masked.drop(columns=spout_cols)

In [ ]:
def plot_tongue_trajectory(dataframe, event_times, clip_length, event_type):
    """Plot tongue x/y and distance-to-spout around each event time.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        Masked tongue table with 'time', 'x', 'y' and 'nearest_spout_distance'.
    event_times : array-like
        Event times to center on, in the same time base as ``dataframe['time']``.
        At most 50.
    clip_length : float
        Width of the plotted window, in seconds.
    event_type : str
        Label for the event marker in the legend.
    """
    if len(event_times) > 50:
        raise ValueError("Cannot plot more than 50 events at once.")

    for event_time in event_times:
        window_start = event_time - clip_length / 2

        fig, axs = plt.subplots(3, 1, figsize=(4, 4), sharex=True)

        line1, = axs[0].plot(dataframe['time'], dataframe['y'], linestyle='-', color='b',
                             label='Kinematic Trajectory')
        axs[0].set_ylabel('Y Position')
        axs[0].set_ylim([200, 400])

        axs[1].plot(dataframe['time'], dataframe['x'], linestyle='-', color='b')
        axs[1].set_ylabel('X Position')
        axs[1].set_ylim([300, 400])

        line2, = axs[2].plot(dataframe['time'], dataframe['nearest_spout_distance'],
                             linestyle='-', color='g', label='Distance to Spout')
        axs[2].set_xlabel('Time (s)')
        axs[2].set_ylabel('Distance to Spout')
        axs[2].set_xlim([window_start, window_start + clip_length])
        axs[2].set_ylim([0, 55])

        line3 = axs[2].axhline(y=30, color='k', ls='--', lw=0.5, label='Spout Threshold')
        line4 = axs[2].axvline(x=event_time, color='r', ls='--', lw=0.5,
                               label='Time of {}'.format(event_type))

        handles = [line1, line2, line3, line4]
        fig.legend(handles, [h.get_label() for h in handles],
                   bbox_to_anchor=(1.05, .65), loc='upper left')

        plt.tight_layout()
        plt.show()


# Lickometer licks with no pose match, and pose licks with no lickometer match
lickometer_only_times = licko_licks_classified.loc[
    licko_licks_classified['Status'] == "False Positive", "Time"].to_numpy()
pose_only_times = LP_licks_classified.loc[
    LP_licks_classified['Status'] == "False Negative", "Time"].to_numpy()

# plot_tongue_trajectory(tongue_masked, pose_only_times[:2], clip_length=1.0,
#                        event_type='pose-only lick')

plot_tongue_trajectory(tongue_masked, lickometer_only_times[:2], clip_length=1.0,
                       event_type='lickometer-only lick')

## 9. Labeled video clips

In [ ]:
# Clip the labeled video at example timepoints to visualize failure modes
timestamps = lickometer_only_times[0:2]
input_video_path = str(DATA / 'video_preds_labeltest/labeled_videos/bottom_camera_labeled.mp4')
clip_length = 1.0  # Clip length in seconds
timestamps_centered = timestamps - clip_length / 2
output_dir = str(SCRATCH / 'labeled_clips' / 'lickometer_only')

extract_clips_ffmpeg_encode(input_video_path, timestamps_centered, clip_length, output_dir)